In [ ]:
from transformers import pipeline
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from nltk.corpus import stopwords
from nltk import download
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from tqdm import tqdm
from collections import Counter
import pandas as pd
import re, os
import hashlib
tqdm.pandas()

In [ ]:
# EXTRACT SOURCE, TARGET, RELATION, AND FULL_TEXT
import pandas as pd

df = pd.read_csv("../sentiment/tweets.csv")

output_dir = "data"
os.makedirs(output_dir, exist_ok=True)

edges = []

for _, row in df.iterrows():
    src = row['username']
    exc = ['grok']
    text = row.get('full_text') or row.get('text')
    normal_text = row.get('normalize')
    stemm_text = row.get('stemming')
    sentiment = row.get('sentiment')

    # === MENTIONED ===
    if pd.notna(row['user_mentions']) and row['user_mentions'] != '-':
        mentions = [m.replace('@', '').strip() for m in row['user_mentions'].split(';') if m.strip()]
        for m in mentions:
            if m != src and m not in exc and src not in exc:
                edges.append({
                    'source': f'@{src}',
                    'target': f'@{m}',
                    'relation': 'mentioned',
                    'full_text': text,
                    'normal_text': normal_text,
                    'stemm_text': stemm_text,
                    'sentiment': sentiment
                })

    # === REPLIED / RETWEETED / QUOTED ===
    if row['relation_type'] in ['replied', 'retweeted', 'quoted']:
        tgt = row['target_username']
        if pd.notna(tgt) and tgt != '-' and tgt.strip() != '' and tgt != src and tgt not in exc and src not in exc:
            edges.append({
                'source': f'@{src}',
                'target': f'@{tgt}',
                'relation': row['relation_type'],
                'full_text': text,
                'normal_text': normal_text,
                'stemm_text': stemm_text,
                'sentiment': sentiment
            })

edges_df = pd.DataFrame(edges, columns=['source', 'target', 'relation', 'full_text','normal_text','stemm_text','sentiment'])

weight_map = {
    'mentioned': 1.5,
    'retweeted': 1.0,
    'quoted': 2.0,
    'replied': 1.5
}

edges_df['weight'] = edges_df['relation'].map(weight_map)
weighted_df = (
    edges_df
    .groupby(['source', 'target'], as_index=False)
    .agg({'weight': 'sum'})
    .rename(columns={'weight': 'total_weight'})
)

edges_df.to_csv(f"{output_dir}/all.csv", index=False)

print(edges_df.info(),'\n')
display(edges_df.head(5))

In [ ]:
df = pd.read_csv(f"{output_dir}/all.csv")

print(f"Total data awal: {len(df)}")

relations = ['mentioned', 'replied', 'quoted', 'retweeted']

for rel in relations:
    subset = df[df['relation'] == rel][['source', 'target', 'full_text','normal_text','stemm_text', 'sentiment']].copy()
    if subset.empty:
        print(f"[WARNING] {rel}: kosong, dilewati.")
        continue

    subset = subset[subset['source'] != subset['target']]

    outpath = f"{output_dir}/{rel}_sentiment.csv"
    subset.to_csv(outpath, index=False)
    print(f"[OK] {rel}: {len(subset)} data disimpan ke {outpath}")

print("[OK] Semua relasi selesai dipisahkan berdasarkan sentimen!")

In [ ]:
embedding_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

def build_stopwords():
    download('stopwords')
    stopword_list = stopwords.words('indonesian')
    extra = [
        'tu', 'uf', 'deh', 'nak', 'amp', 'b', 'a', 'je',
        'sih', 'dos', 'm', 'eh', 'tuh', 'hm', 'nya',
        'ufufuf', 'lho', 'rm', 'ufcc', 'ppv', 'via', 'pon',
        'dok', 'pe', 'gs', 'ya', 'sep', 'go', 'mah',
        'pemerintah','presiden','wakil','prabowo','subianto','gibran',
        'rakabuming','raka','nih','program','indonesia','jalan','pimpin',
        'ri','menteri','hai','perintah','pemerintah','nih hai','anak','berita',
        'lengkap'
    ]
    stopword_list.extend(extra)
    return list(set(stopword_list))

# Generate Summary All

In [ ]:
def process_all(
    input_file: str,
    output_dir: str,
    embedding_model
):
    os.makedirs(output_dir, exist_ok=True)

    df = pd.read_csv(input_file)
    if df.empty:
        print("[WARNING] File kosong, tidak ada data yang diproses.")
        return

    print(f"\nMulai olah seluruh data | {len(df)} baris data")

    # ==== DEGREE ==== #
    edges_unique = df[['source', 'target']].drop_duplicates()
    outdegree = edges_unique['source'].value_counts()
    indegree = edges_unique['target'].value_counts()
    degree = (
        outdegree.add(indegree, fill_value=0)
        .sort_values(ascending=False)
    )

    # ==== WEIGHTED DEGREE ==== #
    edges_weighted = df.groupby(['source', 'target'], as_index=False)['weight'].sum()
    outdegree_w = edges_weighted.groupby('source')['weight'].sum()
    indegree_w = edges_weighted.groupby('target')['weight'].sum()
    degree_w = outdegree_w.add(indegree_w, fill_value=0).sort_values(ascending=False)

    top_users = indegree_w.sort_values(ascending=False).head(5).index.tolist()  # bisa ubah ke 5 kalau mau lebih ringkas
    print(f"Top {len(top_users)} user berdasarkan in-degree:\n{top_users}\n")

    stop_words = build_stopwords()
    topic_summary_list = []

    for user in tqdm(top_users):
        df_user = df[
            (df['relation'].isin(['mentioned', 'replied', 'quoted'])) &
            ((df['target'] == user) | (df['source'] == user))
        ]
        normal_docs = df_user['normal_text'].dropna().tolist()
        stemm_docs = df_user['stemm_text'].dropna().tolist()

        if len(normal_docs) < 15:
            print(f"[WARNING] Skip {user} (dokumen terlalu sedikit: {len(normal_docs)})")
            continue

        print(f"Proses {user}: {len(normal_docs)} tweet | Degree {int(degree.get(user, 0))} | In: {int(indegree.get(user, 0))} | Out: {int(outdegree.get(user, 0))}")

        sentiment_counts = df_user['sentiment'].value_counts(normalize=True).to_dict()
        dominant_sentiment = max(sentiment_counts, key=sentiment_counts.get)
        sentiment_readable = " | ".join([f"{k.capitalize()}: {v*100:.1f}%" for k, v in sentiment_counts.items()])

        # === Topic Modeling (stemm text) === #
        min_topic_size = max(2, min(10, len(stemm_docs)//2))
        vectorizer = CountVectorizer(ngram_range=(1, 2))
        stemm_topic_model = BERTopic(
            embedding_model=embedding_model,
            vectorizer_model=vectorizer,
            min_topic_size=min_topic_size,
            verbose=False,
            nr_topics=min(10, len(stemm_docs)//3)
        )

        topics, probs = stemm_topic_model.fit_transform(stemm_docs)
        topic_info = stemm_topic_model.get_topic_info()

        keywords_all = []
        for _, row in topic_info.iterrows():
            if row['Topic'] != -1 and isinstance(row['Name'], str):
                clean_name = re.sub(r"^\d+_", "", row['Name']).strip()
                words = [w for w in clean_name.split('_') if len(w) > 2 and w not in stop_words]
                keywords_all.extend(words)

        if not keywords_all:
            all_words = []
            for doc in stemm_docs:
                for w in doc.split():
                    if len(w) > 2 and w not in stop_words:
                        all_words.append(w)
            counter = Counter(all_words)
        else:
            counter = Counter(keywords_all)

        top_words = [w for w, _ in counter.most_common(5)]

        # === Topic Modeling (normal text) === #
        min_topic_size = max(2, min(10, len(normal_docs)//2))
        vectorizer = CountVectorizer(stop_words=stop_words, ngram_range=(1, 2))
        normal_topic_model = BERTopic(
            embedding_model=embedding_model,
            vectorizer_model=vectorizer,
            min_topic_size=min_topic_size,
            verbose=False,
            nr_topics=min(10, len(normal_docs)//3)
        )

        normal_topic_model.fit_transform(normal_docs)

        try:
            rep_docs_all = []
            rep_docs = normal_topic_model.get_representative_docs()
            for k, v in rep_docs.items():
                if k != -1:
                    rep_docs_all.extend(v[:3])
        except Exception:
            rep_docs_all = []

        if not rep_docs_all:
            rep_docs_all = normal_docs[:5]

        rep_docs_all = list(dict.fromkeys(rep_docs_all))

        relations_used = df_user['relation'].unique()
        relation_scope = ', '.join(sorted(set(relations_used)))

        topic_summary_list.append({
            'user': user,
            'top_keywords': ' | '.join(sorted(set(top_words))),
            'narasi': ' ... '.join(rep_docs_all[:10]),
            'relation_scope': relation_scope,
            'dominant_sentiment': dominant_sentiment,
            'sentiment_distribution': sentiment_readable,
            'tweet_count': len(normal_docs),
            'degree': int(degree.get(user, 0)),
            'degree_distribution': f"in: {int(indegree.get(user, 0))} | out: {int(outdegree.get(user, 0))}",
            'weighted_degree': float(degree_w.get(user, 0)),
            'w_degree_distribution': f"in: {float(indegree_w.get(user, 0))} | out: {float(outdegree_w.get(user, 0))}"
        })

    # === Simpan hasil === #
    result_df = pd.DataFrame(topic_summary_list)
    result_df = result_df.sort_values(by='tweet_count', ascending=False)
    outpath = f"{output_dir}/all_summary.csv"
    result_df.to_csv(outpath, index=False, encoding='utf-8-sig')

    print(f"\n[OK] Selesai olah all.csv — hasil disimpan ke: {outpath}")
    return result_df

process_all(
    input_file="data/all.csv",
    output_dir="bertopic_results",
    embedding_model=embedding_model
)

# Generate Summary based on relation

In [ ]:
def process_relation(
    relation: str,
    input_dir: str,
    output_dir: str,
    embedding_model
):
    os.makedirs(output_dir, exist_ok=True)

    file_path = f"{input_dir}/{relation}_sentiment.csv"
    df = pd.read_csv(file_path)

    if df.empty:
        print(f"[WARNING] File kosong: {relation}")
        return

    # ==== DEGREE ==== #
    edges_unique = df[['source', 'target']].drop_duplicates()
    outdegree = edges_unique['source'].value_counts()
    indegree = edges_unique['target'].value_counts()
    degree = (
        outdegree.add(indegree, fill_value=0)
        .sort_values(ascending=False)
    )

    top_users = indegree.head(5).index.tolist()

    print(f"\nMulai olah relasi: {relation.upper()} | {len(df)} baris data")

    stop_words = build_stopwords()
    topic_summary_list = []

    for user in tqdm(top_users):
        df_user = df[(df['target'] == user) | (df['source'] == user)]
        normal_docs = df_user['normal_text'].dropna().tolist()
        stemm_docs = df_user['stemm_text'].dropna().tolist()

        if len(normal_docs) < 15:
            print(f"[WARNING] Skip {user} (dokumen terlalu sedikit: {len(normal_docs)})")
            continue

        print(f"\nProses {user}: {len(normal_docs)} tweet | Degree {int(degree.get(user, 0))} | In-degree {int(indegree.get(user, 0))} | Out-degree {int(outdegree.get(user, 0))}")

        sentiment_counts = df_user['sentiment'].value_counts(normalize=True).to_dict()
        dominant_sentiment = max(sentiment_counts, key=sentiment_counts.get)
        sentiment_readable = " | ".join([f"{k.capitalize()}: {v*100:.1f}%" for k, v in sentiment_counts.items()])

        min_topic_size = max(2, min(10, len(stemm_docs)//2))
        vectorizer = CountVectorizer(ngram_range=(1, 2))
        stemm_topic_model = BERTopic(
            embedding_model=embedding_model,
            vectorizer_model=vectorizer,
            min_topic_size=min_topic_size,
            verbose=False,
            nr_topics=min(10, len(stemm_docs)//3)
        )

        topics, probs = stemm_topic_model.fit_transform(stemm_docs)
        topic_info = stemm_topic_model.get_topic_info()

        keywords_all = []
        for _, row in topic_info.iterrows():
            if row['Topic'] != -1 and isinstance(row['Name'], str):
                clean_name = re.sub(r"^\d+_", "", row['Name']).strip()
                words = [w for w in clean_name.split('_') if len(w) > 2 and w not in stop_words]
                keywords_all.extend(words)

        if not keywords_all:
            all_words = []
            for doc in stemm_docs:
                for w in doc.split():
                    if len(w) > 2 and w not in stop_words:
                        all_words.append(w)
            counter = Counter(all_words)
        else:
            counter = Counter(keywords_all)

        top_words = [w for w, _ in counter.most_common(5)]

        min_topic_size = max(2, min(10, len(normal_docs)//2))
        vectorizer = CountVectorizer(stop_words=stop_words, ngram_range=(1, 2))
        normal_topic_model = BERTopic(
            embedding_model=embedding_model,
            vectorizer_model=vectorizer,
            min_topic_size=min_topic_size,
            verbose=False,
            nr_topics=min(10, len(normal_docs)//3)
        )

        normal_topic_model.fit_transform(normal_docs)

        try:
            rep_docs_all = []
            rep_docs = normal_topic_model.get_representative_docs()
            for k, v in rep_docs.items():
                if k != -1:
                    rep_docs_all.extend(v[:3])

        except Exception:
            rep_docs_all = []

        if not rep_docs_all:
            rep_docs_all = normal_docs[:5]

        rep_docs_all = list(dict.fromkeys(rep_docs_all))

        topic_summary_list.append({
            'user': user,
            'top_keywords': ' | '.join(sorted(set(top_words))),
            'narasi': ' ... '.join(rep_docs_all[:10]),
            'relation_scope': relation,
            'dominant_sentiment': dominant_sentiment,
            'sentiment_distribution': sentiment_readable,
            'tweet_count': len(normal_docs),
            'degree': int(degree.get(user, 0)),
            'degree_distribution': f"in: {int(indegree.get(user, 0))} | out: {int(outdegree.get(user, 0))}"
        })

    # ==== Simpan Hasil ==== #
    result_df = pd.DataFrame(topic_summary_list)
    result_df = result_df.sort_values(by='tweet_count', ascending=False)
    outpath = f"{output_dir}/{relation}_summary.csv"
    result_df.to_csv(outpath, index=False, encoding='utf-8-sig')

    print(f"\n[OK] {relation.upper()} selesai — hasil disimpan ke: {outpath}")
    return result_df

process_relation(
    relation="mentioned",
    input_dir="data",
    output_dir="bertopic_results",
    embedding_model=embedding_model
)